In [1]:
import pandas as pd
import numpy as np

In [4]:
RequestAccepted = pd.DataFrame({
    "requester_id": [1, 1, 2, 3],
    "accepter_id": [2, 3, 3, 4],
    "accept_date": ["2016/06/03", "2016/06/08", "2016/06/08", "2016/06/09"]
})

In [22]:
req_count = RequestAccepted.groupby('requester_id')[['accepter_id']].count().rename(columns={'accepter_id':'req_count'}).reset_index()

In [23]:
req_count

,requester_id,req_count
0,1,2
1,2,1
2,3,1


In [20]:
acp_count = RequestAccepted.groupby('accepter_id')[['requester_id']].count().rename(columns={'requester_id':'acp_count'}).reset_index()

In [21]:
acp_count

,accepter_id,acp_count
0,2,1
1,3,2
2,4,1


In [35]:
merged = req_count.merge(acp_count, left_on='requester_id', right_on='accepter_id', how='outer')

In [45]:
merged.fillna(0, inplace=True)

In [46]:
merged['total_count'] = merged['req_count'] + merged['acp_count']

In [47]:
merged

,requester_id,req_count,accepter_id,acp_count,total_count
0,1.0,2.0,0.0,0.0,2.0
1,2.0,1.0,2.0,1.0,2.0
2,3.0,1.0,3.0,2.0,3.0
3,0.0,0.0,4.0,1.0,1.0


In [49]:
row = merged[merged['total_count'] == merged.total_count.max()]

In [57]:
row.reset_index(inplace=True)

In [60]:
id = row.loc[0, 'requester_id'] or row.loc[0, 'accpeter_id']

In [61]:
count = row.loc[0, 'total_count']

In [62]:
pd.DataFrame({'id':[id], 'num':[count]})

,id,num
0,3.0,3.0


In [79]:
tree = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "p_id": [np.nan, 1, 1, 2, 2]
})

In [80]:
tree_ext = tree.merge(tree, left_on='id', right_on='p_id', how='left').\
        drop('p_id_y', axis=1).rename(columns={'id_x': 'id','id_y':'c_id', 'p_id_x':'p_id'})

In [81]:
tree_ext

,id,p_id,c_id
0,1,NaN,2.0
1,1,NaN,3.0
2,2,1.0,4.0
3,2,1.0,5.0
4,3,1.0,NaN
5,4,2.0,NaN
6,5,2.0,NaN


In [85]:
tree_ext.drop_duplicates('id', inplace=True)

In [87]:
tree_ext['type'] = 'Inner'

In [97]:
tree_ext.loc[tree_ext['p_id'].isna(), 'type'] = 'Root'

In [99]:
tree_ext.loc[tree_ext['c_id'].isna(), 'type'] = 'Leaf'

In [100]:
tree_ext

,id,p_id,c_id,type
0,1,NaN,2.0,Root
2,2,1.0,4.0,Inner
4,3,1.0,NaN,Leaf
5,4,2.0,NaN,Leaf
6,5,2.0,NaN,Leaf


In [114]:
df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "student": ["Abbot", "Doris", "Emerson", "Green", "Jeames"]
})

In [115]:

df

,id,student
0,1,Abbot
1,2,Doris
2,3,Emerson
3,4,Green
4,5,Jeames


In [116]:
def swap(x):
    res = {}
    for i in x.index:
        if x[i] % 2 == 0:
            res[i] = x[i] - 1
        elif i != len(x.index) - 1:
            res[i] = x[i] + 1
        else:
            res[i] = x[i]
    return pd.Series(res)
            
df['id'] = df[['id']].apply(swap)

i:  4
x[i]:  5


In [118]:
df.sort_values('id')

,id,student
1,1,Doris
0,2,Abbot
3,3,Green
2,4,Emerson
4,5,Jeames


In [163]:
user_content = pd.DataFrame({
    "content_id": [1, 2, 3, 4],
    "content_text": [
        "hello world of SQL",
        "the QUICK-brown fox",
        "modern-day DATA science",
        "web-based FRONT-end development"
    ]
})

In [168]:
user_content

,content_id,content_text
0,1,Hello World Of Sql
1,2,The Quick-Brown Fox
2,3,Modern-Day Data Science
3,4,Web-Based Front-End Development


In [169]:
def string_process(content):
    content = content.replace('-', '- ').lower()
    words = content.split()
    res = []
    for word in words:
        res.append(word[0].upper() + word[1:])
    return " ".join(res).replace('- ', '-')

In [173]:
user_content['converted_text'] = ''
for i in range(len(user_content)):
    content = user_content.iloc[i]['content_text']
    updated_content = string_process(content)
    user_content.loc[user_content.index[i], 'converted_text'] = updated_content
user_content.rename(columns={'content_text':'original_text'}, inplace=True)

In [174]:
user_content

,content_id,original_text,converted_text
0,1,Hello World Of Sql,Hello World Of Sql
1,2,The Quick-Brown Fox,The Quick-Brown Fox
2,3,Modern-Day Data Science,Modern-Day Data Science
3,4,Web-Based Front-End Development,Web-Based Front-End Development


In [148]:
content = user_content.iloc[3]['content_text']

In [152]:
content = content.replace('-', '- ').lower()

In [153]:
words = content.split()

In [155]:
res = []
for word in words:
    res.append(word[0].upper() + word[1:])

In [158]:
" ".join(res).replace('- ', '-')

'Web-Based Front-End Development'

In [199]:
logs = pd.DataFrame({
    "log_id": [1, 2, 3, 4, 5, 6, 7],
    "ip": [
        "192.168.1.1",
        "256.1.2.3",
        "192.168.001.1",
        "192.168.1.1",
        "192.168.1",
        "256.1.2.3",
        "192.168.001.1"
    ],
    "status_code": [200, 404, 200, 200, 500, 404, 200]
})

In [200]:
logs

,log_id,ip,status_code
0,1,192.168.1.1,200
1,2,256.1.2.3,404
2,3,192.168.001.1,200
3,4,192.168.1.1,200
4,5,192.168.1,500
5,6,256.1.2.3,404
6,7,192.168.001.1,200


In [201]:
counts = logs.groupby('ip')['log_id'].count().reset_index()

In [202]:
counts['valid'] = True
for i in range(len(counts)):
    ip = counts.iloc[i]['ip']
    valid = check_valid(ip)
    counts.loc[counts.index[i], 'valid'] = valid

valid:  False
valid:  False
valid:  True
valid:  False


In [207]:
counts[counts['valid'] == False][['ip', 'log_id']].sort_values(by=['log_id', 'ip'], ascending=False).rename(columns={'log_id':'invalid_count'})

,ip,invalid_count
3,256.1.2.3,2
0,192.168.001.1,2
1,192.168.1,1


In [188]:
def check_valid(ip):
    parts = ip.split('.')
    if len(parts) !=4:
        return False
    for part in parts:
        if not part or part[0] == '0':
            return False
        num = int(part)
        if num >= 256 or num < 0:
            return False
    return True

In [211]:
data = [[1, 'Alice', None, 12000, 'Executive'], [2, 'Bob', 1, 10000, 'Sales'], [3, 'Charlie', 1, 10000, 'Engineering'], [4, 'David', 2, 7500, 'Sales'], [5, 'Eva', 2, 7500, 'Sales'], [6, 'Frank', 3, 9000, 'Engineering'], [7, 'Grace', 3, 8500, 'Engineering'], [8, 'Hank', 4, 6000, 'Sales'], [9, 'Ivy', 6, 7000, 'Engineering'], [10, 'Judy', 6, 7000, 'Engineering']]
employees = pd.DataFrame(columns=["employee_id", "employee_name", "manager_id", "salary", "department"]).astype({"employee_id": "int", "employee_name": "string", "manager_id": "Int64", "salary": "int", "department": "string"})
employees = pd.DataFrame(
    data,
    columns=["employee_id", "employee_name", "manager_id", "salary", "department"]
).astype({
    "employee_id": "int",
    "employee_name": "string",
    "manager_id": "Int64",
    "salary": "int",
    "department": "string"
})

In [215]:
employees

,employee_id,employee_name,manager_id,salary,department
0,1,Alice,<NA>,12000,Executive
1,2,Bob,1,10000,Sales
2,3,Charlie,1,10000,Engineering
3,4,David,2,7500,Sales
4,5,Eva,2,7500,Sales
5,6,Frank,3,9000,Engineering
6,7,Grace,3,8500,Engineering
7,8,Hank,4,6000,Sales
8,9,Ivy,6,7000,Engineering
9,10,Judy,6,7000,Engineering


In [219]:
boss_id = employees[employees.manager_id.isna()].employee_id.iloc[0]

In [220]:
from collections import defaultdict
reports = defaultdict(list)
team_size, level, budget = defaultdict(int), defaultdict(int), defaultdict(int)

In [223]:
for i in range(len(employees)):
    row = employees.iloc[i]
    if pd.notna(row.manager_id):
        mgr = row.manager_id

1
1
2
2
3
3
4
6
6
